# 04 · H3: чинит ли steering разрыв, и не дешевле ли альтернатива

Полный цикл: база → probe → вектор на train → подбор на val → заморозка → test
со сравнениями (сильный промпт, пересказ ответа) и sham-контролями
(случайное направление, перестановка координат, соседний слой, окно до ответа).

Запускать после 03, когда H1 и H2 подтверждены. Все решения базы и активации
берутся из кэша 03 — пересчитывается только новое.

> **Протокол** (`docs/HYPOTHESES.md`, `AGENTS.md`): вектор строится только на train,
> подбор — только на val, test открывается **один раз** на замороженный метод.
> Пайплайн это проверяет сам — если ячейка упала с `TestLockError` или `StageOrderError`,
> это защита сработала, а не баг.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Mac: неподдержанные операции -> CPU

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

from steering_tool_feedback.config import ExperimentConfig
from steering_tool_feedback.pipeline import Experiment
from steering_tool_feedback import journal, viz

In [ ]:
cfg = ExperimentConfig.make(
    name="h3-steering-qwen3-1.7b", hypothesis="H3", notebook="notebooks/04_h3_steering.ipynb",
    notes="Steering по CAA из верных/неверных решений базы против промпта и пересказа.",
    overrides={
        "model.name": "Qwen/Qwen3-1.7B",
        "data.n_tasks": 160,
        "eval.methods": ["baseline", "strong_prompt", "restate", "steering"],
        "steer.alphas": [0.25, 0.5, 1.0, 2.0, 4.0],
    },
)
exp = Experiment(cfg)
exp.data()
exp.estimate()

In [ ]:
exp.run("sweep")
viz.dose_response(exp.sweep_table(), cfg.steer.max_specificity_drop);
exp.state["selection"]

### Test — один раз. Дальше этой ячейки конфиг не меняется.

In [ ]:
exp.freeze()
exp.test()
viz.compare_methods(exp.results_table("test"));
exp.results_table("test")

In [ ]:
for h, v in exp.verdicts().items():
    print(f"{h}: {v.label:16} {v.reason}")

In [ ]:
display(Markdown(exp.report(
    conclusion="""…""",
    next_step="""…""",
)))